# Graphviz setup

in jupyter console
```sh
gem install --user-install ruby-graphviz
```

and the man himself via Docker shell
```sh
# 1. Find the container name/ID:
docker ps

# 2. Run apt as root inside the running container:
docker exec -u 0 -it <container_name_or_id> /bin/bash

# 3. Inside that root shell:
apt-get update
apt-get install -y graphviz
exit
```

> sometimes manually restart/reselect kernel in Jupyter notebook?

In [51]:
require_relative 'draw_dot'

false

# MICROGRAD in Ruby

raw doging micrograd python demo but in Ruby
https://www.youtube.com/watch?v=VMj-3S1tku0&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ

In [319]:
require 'set'

class Value
    attr_reader :data, :children, :op
    attr_accessor :label, :grad, :backward

    def initialize(data, children = [], operator = '', label: '')
        @data = data
        @grad = 0.0
        @backward = -> {}
        @children = children.to_set # consider other data structures?
        @op = operator
        @label = label
    end

    def inspect
        # similar to python just for the lulz
        "Value:(data=#{@data})"
    end

    def +(operand)
        out = Value.new( self.data + operand.data, [self, operand], '+')
        out.backward = lambda {
            self.grad += 1.0 * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += 1.0 *out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    
    def *(operand)
        out = Value.new( self.data * operand.data, [self, operand], '*')
        out.backward = lambda {
            self.grad += operand.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += self.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    # activation(squashing) func - can be any hyperbolic function to produce -1...1 range
    def tanh
        x = self.data
        t = (Math.exp(2 * x) - 1) / (Math.exp(2 * x) + 1)
        out = Value.new(t, [self], 'tanh - squashing/activation func')

        out.backward = lambda {
            self.grad = (1 - out.data**2) * out.grad
        }
        out
    end

end


:tanh

In [156]:
a = Value.new(2.0, label: 'a')
b = Value.new(-3.0, label: 'b')
c = Value.new(10.0, label: 'c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value.new(-2.0, label: 'f')
l = f * d; l.label = 'L' # making it lower case as 'L' will be treated as const

"L"

In [157]:
d.children

#<Set: {Value:(data=-6.0), Value:(data=10.0)}>

In [158]:
d.children.first.children

#<Set: {Value:(data=2.0), Value:(data=-3.0)}>

In [159]:
d.op

"+"


# Backpropagation

we start at L and calc **gradient** along the nodes.

**gradient** - derivative of each node with respect to L.

In Neural Network we are interested in **"Loss Function"** - L (later final output axon) with respect to all these **weights (and later weights+inputs)** - nodes 'a - f'. So we can see how we can tweak each node to affect L, the output axon signal

> refer Py notes for manual breakdown of derivatives(grad)

In [160]:
d.grad = -2.0  # value of 'f'
f.grad = 4.0  # value of 'd'
l.grad = 1.0

c.grad = -2.0
e.grad = -2.0

a.grad = 6.0
b.grad = -4.0

-4.0

In [161]:
g = draw_dot_graphviz(l)
g.output(svg: "micrograd.svg")

![micrograd.svg](micrograd.svg)

# Neurons

![image](https://www.researchgate.net/publication/364814302/figure/fig5/AS:11431281092677232@1666928276027/Neural-net-Structure-with-an-Activation-Function-Source-CS231n-Stanford-2017.png)


[cs231n neuron images](https://www.google.com/search?sca_esv=5aba584b8a922c2c&udm=2&fbs=ADc_l-aN0CWEZBOHjofHoaMMDiKpaEWjvZ2Py1XXV8d8KvlI3p-ML-906rRL_m6h4jR-tdCH-vUIlZq9RzugLEcfjf51b4dfDKizXS4hTwRCZW2Tyeo_h9FK7Iw3R0k8aayAiwizhG1xI1JXZfQwT07KT0cm5LxxccrVaZNLTb_5H1XATdGr7j71tLMKK8t_5Ec039O8ZXGxZyM9qt7SU8VWruIU_kGXeQ&q=cs231n+neuron&sa=X&ved=2ahUKEwiwqIawxLmTAxVCRWcHHfCSFJYQtKgLegQIDhAB&biw=1600&bih=1658&dpr=2#sv=CAMSVhoyKhBlLV80VnZNTkt2aWdESkdNMg5fNFZ2TU5LdmlnREpHTToOQTVSNkMyaVNESUljak0gBCocCgZtb3NhaWMSEGUtXzRWdk1OS3ZpZ0RKR00YADABGAcg_J7elQswAkoIEAEYASABKAE)

---

- 'w' are weights
- 'b'  in cell is 'bias'
1. we cumulate all the input with weights and add bias
2. and we produce activations func(usually some squashing func, i.e. sigmoid of tanH func)

## Neuron in Ruby

In [298]:
# inputs x
x1 = Value.new(2.0, label: 'axon signal from neuron x1')
x2 = Value.new(0.0, label: 'axon signal from neuron  x2')
# weights w
w1 = Value.new(-3.0, label: 'synapse wight w1')
w2 = Value.new(1.0, label: 'synapse wight w2')
# bias of neuron
b = Value.new(6.8813, label: 'Neuron Bias')

# input signals multiplied by the weights of the dendrite they were received
x1w1 = x1*w1; x1w1.label = 'dendrite signal x1w1'
x2w2 = x2*w2; x2w2.label = 'dendrite signal x2w2'

# then they cumulated
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'Sum x1w1x2w2'
# then bias is added to get the BODY OF NEURON, without activation func (output axon)
n = x1w1x2w2 + b; n.label = 'NEURON'

"NEURON"

## Activation function
now let's add squashing function(i.e. tanH) to *Value obj.* to get ***activation function* for the output axon**

[https://en.wikipedia.org/wiki/Hyperbolic_functions#Definitions](https://en.wikipedia.org/wiki/Hyperbolic_functions#Definitions )

```rb
def tanh
  x = @data
  t = (Math.exp(2 * x) - 1) / (Math.exp(2 * x) + 1)
  Value.new(t, [self], 'tanh')
end
```

In [299]:
o = n.tanh; o.label = 'output O'

"output O"

# Backpropagation of Neuron

let's add **'backward'** lambdas to all operations to automatically get derivatives/ gradients (back propagate).

refer Python notes for pen and paper calc. 

> for tanH derivative we used [cheat sheet](https://en.wikipedia.org/wiki/Hyperbolic_functions#Derivatives)
>>>
```rb
1 - tan_h_result.data**2
```

## Semi-automatic propagation

where we manually sequence

In [222]:
o.grad = 1.0
o.backward.call
n.backward.call

x1w1x2w2.backward.call
# # b_backward.call # nothing will happen as it is a leaf node

x1w1.backward.call
x2w2.backward.call

0.0

In [228]:
graph_o = draw_dot_graphviz(o)
graph_o.output(svg: "neuron.svg")

![neuron.svg](neuron.svg)


![neuron](https://image1.slideserve.com/2574549/nerve-impulses-l.jpg)

## Sorting for fully automatic back propagation

but ideally before we propagate, we need to sort the tree nodes using [https://en.wikipedia.org/wiki/Topological_sorting](https://en.wikipedia.org/wiki/Topological_sorting), so we propagate only in safe order: from output back to leaves.

In [318]:
# note: 'propagate_all' can be run once, otherwise will be accumulating values
Value.class_eval do
    def propagate_all
      topo = []
      visited = Set.new

      build_topo = lambda do |v|
        next if visited.include?(v)

        visited.add(v)
        v.children.each { |child| build_topo.call(child) }
        topo << v
      end

      build_topo.call(self)

      @grad = 1.0
      topo.reverse_each { |v| v.backward.call }
    end
end

o.propagate_all.map { _1.grad }
# o.propagate_all.map { _1.grad = 0.0 }

[-1.5001561057025832, 1.0001040704683888, 0.5000520352341944, 0.5000520352341944, 0.0, 0.5000520352341944, 0.5000520352341944, 0.5000520352341944, 0.5000520352341944, 1.0]

In [310]:
auto_graph_o = draw_dot_graphviz(o)
auto_graph_o.output(svg: "neuron.svg")

In [ ]:
a = Value.new(2.0, label: 'a')